In [3]:
# ==========================================================
# NHANES 2017–March 2020 pre-pandemic
# Diabetes: PR model + ZI comparator
#   - non-shared comparison: OBS_ZI vs PR
#   - shared comparison:     OBS_ZI vs PR
#
# Main outcome:
#   Y = self-reported diabetes from DIQ010
#
# Reveal proxy:
#   A1C: LBXGH >= 6.5  (primary; WTMECPRP)
#   FPG: LBXGLU >= 126 (sensitivity; WTSAFPRP)
#
# Notes:
#   * non-shared design avoids relabeling issues
#   * shared design uses d-only ridge stabilization
#   * SEs are Hessian-based model SEs, not full survey-design SEs
# ==========================================================

import os, warnings
import numpy as np
import pandas as pd
from scipy.special import expit
from scipy.optimize import minimize
from statsmodels.tools.numdiff import approx_hess

warnings.filterwarnings("ignore")
np.set_printoptions(suppress=True, precision=6)

# -----------------------------
# settings
# -----------------------------
ADULT_AGE = 20
EPS = 1e-10
MAXITER = 4000
BOUND = 10.0
SEED = 123
RIDGE_LAMBDA_SHARED = 0.50
N_STARTS_SHARED = 24
JITTER_SD_SHARED = 0.35

# non-shared design
XA_COLS = ["const", "INSURED", "USUALCARE", "AGE_YEARS_Z", "BMI_Z", "FEMALE"]
XB_COLS = ["const", "AGE_YEARS_Z", "BMI_Z", "FEMALE"]

# shared design
XS_COLS = ["const", "INSURED", "USUALCARE", "AGE_YEARS_Z", "BMI_Z", "FEMALE"]

OUTDIR = "/content/work/nhanes_diabetes_pr_and_zi"
os.makedirs(OUTDIR, exist_ok=True)

# -----------------------------
# helpers
# -----------------------------
def read_xpt(url):
    return pd.read_sas(url, format="xport")

def clip01(x, eps=EPS):
    return np.clip(x, eps, 1.0 - eps)

def weighted_mean(x, w):
    m = np.isfinite(x) & np.isfinite(w)
    return np.sum(w[m] * x[m]) / np.sum(w[m])

def fit_weighted_logit(X, y, w):
    def nll(th):
        p = clip01(expit(X @ th))
        return -np.sum(w * (y * np.log(p) + (1 - y) * np.log(1 - p)))
    def grad(th):
        p = expit(X @ th)
        return X.T @ (w * (p - y))
    r = minimize(
        nll, np.zeros(X.shape[1]),
        jac=grad,
        method="L-BFGS-B",
        bounds=[(-BOUND, BOUND)] * X.shape[1],
        options={"maxiter": MAXITER}
    )
    return r.x

# -----------------------------
# non-shared OBS-ZI
# -----------------------------
def obs_ns_nll(par, Xa, Xb, y, w):
    pa = Xa.shape[1]
    a = par[:pa]
    b = par[pa:]
    pi = clip01(expit(Xa @ a))
    mu = clip01(expit(Xb @ b))
    q = clip01(1 - pi * mu)
    ll = y * (np.log(pi) + np.log(mu)) + (1 - y) * np.log(q)
    return -np.sum(w * ll)

def obs_ns_grad(par, Xa, Xb, y, w):
    pa = Xa.shape[1]
    a = par[:pa]
    b = par[pa:]
    pi = clip01(expit(Xa @ a))
    mu = clip01(expit(Xb @ b))
    q = clip01(1 - pi * mu)

    ga = y * (1 - pi) - (1 - y) * (mu * pi * (1 - pi)) / q
    gb = y * (1 - mu) - (1 - y) * (pi * mu * (1 - mu)) / q

    return -np.concatenate([
        Xa.T @ (w * ga),
        Xb.T @ (w * gb)
    ])

def fit_obs_nonshared(Xa, Xb, y, w):
    a0 = fit_weighted_logit(Xa, y, w)
    b0 = fit_weighted_logit(Xb, y, w)
    start = np.concatenate([a0, b0])

    r = minimize(
        obs_ns_nll, start,
        args=(Xa, Xb, y, w),
        jac=obs_ns_grad,
        method="L-BFGS-B",
        bounds=[(-BOUND, BOUND)] * len(start),
        options={"maxiter": MAXITER}
    )

    H = approx_hess(r.x, lambda th: obs_ns_nll(th, Xa, Xb, y, w))
    cov = np.linalg.pinv(H)
    se = np.sqrt(np.clip(np.diag(cov), 0, np.inf))

    pa = Xa.shape[1]
    return {
        "success": bool(r.success),
        "nll": float(r.fun),
        "par": r.x,
        "cov": cov,
        "se": se,
        "alpha": r.x[:pa],
        "beta": r.x[pa:],
        "se_alpha": se[:pa],
        "se_beta": se[pa:],
        "cond_hess": float(np.linalg.cond(H))
    }

# -----------------------------
# non-shared PR
# -----------------------------
def pr_ns_nll(par, Xa, Xb, y, rev_pos, w):
    pa = Xa.shape[1]
    a = par[:pa]
    b = par[pa:]

    pi = clip01(expit(Xa @ a))
    mu = clip01(expit(Xb @ b))
    q = clip01(1 - pi * mu)

    idx1 = (y == 1)
    idxr = (rev_pos == 1)
    idx0 = ~(idx1 | idxr)

    ll = np.zeros_like(y, dtype=float)
    ll[idx1] = np.log(pi[idx1]) + np.log(mu[idx1])
    ll[idxr] = np.log(pi[idxr]) + np.log(1 - mu[idxr])
    ll[idx0] = np.log(q[idx0])

    return -np.sum(w * ll)

def pr_ns_grad(par, Xa, Xb, y, rev_pos, w):
    pa = Xa.shape[1]
    a = par[:pa]
    b = par[pa:]

    pi = clip01(expit(Xa @ a))
    mu = clip01(expit(Xb @ b))
    q = clip01(1 - pi * mu)

    idx1 = (y == 1)
    idxr = (rev_pos == 1)
    idx0 = ~(idx1 | idxr)

    ga = np.zeros_like(y, dtype=float)
    gb = np.zeros_like(y, dtype=float)

    ga[idx1] = 1 - pi[idx1]
    gb[idx1] = 1 - mu[idx1]

    ga[idxr] = 1 - pi[idxr]
    gb[idxr] = -mu[idxr]

    ga[idx0] = -(mu[idx0] * pi[idx0] * (1 - pi[idx0])) / q[idx0]
    gb[idx0] = -(pi[idx0] * mu[idx0] * (1 - mu[idx0])) / q[idx0]

    return -np.concatenate([
        Xa.T @ (w * ga),
        Xb.T @ (w * gb)
    ])

def fit_pr_nonshared(Xa, Xb, y, rev_pos, w):
    a0 = fit_weighted_logit(Xa, y, w)
    b0 = fit_weighted_logit(Xb, y, w)
    start = np.concatenate([a0, b0])

    r = minimize(
        pr_ns_nll, start,
        args=(Xa, Xb, y, rev_pos, w),
        jac=pr_ns_grad,
        method="L-BFGS-B",
        bounds=[(-BOUND, BOUND)] * len(start),
        options={"maxiter": MAXITER}
    )

    H = approx_hess(r.x, lambda th: pr_ns_nll(th, Xa, Xb, y, rev_pos, w))
    cov = np.linalg.pinv(H)
    se = np.sqrt(np.clip(np.diag(cov), 0, np.inf))

    pa = Xa.shape[1]
    return {
        "success": bool(r.success),
        "nll": float(r.fun),
        "par": r.x,
        "cov": cov,
        "se": se,
        "alpha": r.x[:pa],
        "beta": r.x[pa:],
        "se_alpha": se[:pa],
        "se_beta": se[pa:],
        "cond_hess": float(np.linalg.cond(H))
    }

# -----------------------------
# shared models with d-only ridge
# alpha = m+d, beta = m-d
# -----------------------------
def unpack_md(z, p):
    m = z[:p]
    d = z[p:]
    alpha = m + d
    beta  = m - d
    return m, d, alpha, beta

def shared_obs_nll(z, X, y, w, lam):
    p = X.shape[1]
    m, d, a, b = unpack_md(z, p)
    pa = clip01(expit(X @ a))
    pb = clip01(expit(X @ b))
    pm = clip01(pa * pb)
    ll = y * (np.log(pa) + np.log(pb)) + (1 - y) * np.log(1 - pm)
    pen = 0.5 * lam * np.sum(d * d)
    return -np.sum(w * ll) + pen

def shared_obs_grad(z, X, y, w, lam):
    p = X.shape[1]
    m, d, a, b = unpack_md(z, p)
    pa = clip01(expit(X @ a))
    pb = clip01(expit(X @ b))
    q = clip01(1 - pa * pb)

    ga = y * (1 - pa) - (1 - y) * (pb * pa * (1 - pa)) / q
    gb = y * (1 - pb) - (1 - y) * (pa * pb * (1 - pb)) / q

    gm = X.T @ (w * (ga + gb))
    gd = X.T @ (w * (ga - gb))

    return -np.concatenate([gm, gd]) + np.concatenate([np.zeros(p), lam * d])

def shared_pr_nll(z, X, y, rev_pos, w, lam):
    p = X.shape[1]
    m, d, a, b = unpack_md(z, p)
    pa = clip01(expit(X @ a))
    pb = clip01(expit(X @ b))
    q = clip01(1 - pa * pb)

    idx1 = (y == 1)
    idxr = (rev_pos == 1)
    idx0 = ~(idx1 | idxr)

    ll = np.zeros_like(y, dtype=float)
    ll[idx1] = np.log(pa[idx1]) + np.log(pb[idx1])
    ll[idxr] = np.log(pa[idxr]) + np.log(1 - pb[idxr])
    ll[idx0] = np.log(q[idx0])

    pen = 0.5 * lam * np.sum(d * d)
    return -np.sum(w * ll) + pen

def shared_pr_grad(z, X, y, rev_pos, w, lam):
    p = X.shape[1]
    m, d, a, b = unpack_md(z, p)
    pa = clip01(expit(X @ a))
    pb = clip01(expit(X @ b))
    q = clip01(1 - pa * pb)

    idx1 = (y == 1)
    idxr = (rev_pos == 1)
    idx0 = ~(idx1 | idxr)

    ga = np.zeros_like(y, dtype=float)
    gb = np.zeros_like(y, dtype=float)

    ga[idx1] = 1 - pa[idx1]
    gb[idx1] = 1 - pb[idx1]

    ga[idxr] = 1 - pa[idxr]
    gb[idxr] = -pb[idxr]

    ga[idx0] = -(pb[idx0] * pa[idx0] * (1 - pa[idx0])) / q[idx0]
    gb[idx0] = -(pa[idx0] * pb[idx0] * (1 - pb[idx0])) / q[idx0]

    gm = X.T @ (w * (ga + gb))
    gd = X.T @ (w * (ga - gb))

    return -np.concatenate([gm, gd]) + np.concatenate([np.zeros(p), lam * d])

def raw_shared_obs_nll_from_ab(alpha, beta, X, y, w):
    pa = clip01(expit(X @ alpha))
    pb = clip01(expit(X @ beta))
    pm = clip01(pa * pb)
    ll = y * (np.log(pa) + np.log(pb)) + (1 - y) * np.log(1 - pm)
    return -np.sum(w * ll)

def raw_shared_pr_nll_from_ab(alpha, beta, X, y, rev_pos, w):
    pa = clip01(expit(X @ alpha))
    pb = clip01(expit(X @ beta))
    q = clip01(1 - pa * pb)

    idx1 = (y == 1)
    idxr = (rev_pos == 1)
    idx0 = ~(idx1 | idxr)

    ll = np.zeros_like(y, dtype=float)
    ll[idx1] = np.log(pa[idx1]) + np.log(pb[idx1])
    ll[idxr] = np.log(pa[idxr]) + np.log(1 - pb[idxr])
    ll[idx0] = np.log(q[idx0])

    return -np.sum(w * ll)

def make_starts_shared(anchor, n_starts=24, jitter_sd=0.35, seed=123):
    rng = np.random.default_rng(seed)
    p = len(anchor)
    starts = []

    alpha0 = anchor.copy(); beta0 = np.zeros(p)
    m0 = 0.5 * (alpha0 + beta0); d0 = 0.5 * (alpha0 - beta0)
    starts.append(np.r_[m0, d0])
    starts.append(np.r_[m0, -d0])

    alpha0 = anchor.copy(); beta0 = anchor.copy()
    m0 = 0.5 * (alpha0 + beta0); d0 = 0.5 * (alpha0 - beta0)
    starts.append(np.r_[m0, d0])

    starts.append(np.zeros(2 * p))

    for _ in range(n_starts - len(starts)):
        alpha0 = anchor + rng.normal(scale=jitter_sd, size=p)
        beta0 = rng.normal(scale=jitter_sd, size=p)
        m0 = 0.5 * (alpha0 + beta0)
        d0 = 0.5 * (alpha0 - beta0)
        if rng.uniform() < 0.5:
            d0 = -d0
        starts.append(np.r_[m0, d0])

    return starts

def fit_multistart_shared(obj, grad, X, y, w, starts, rev_pos=None, lam=0.3):
    best = None
    for s in starts:
        if rev_pos is None:
            args = (X, y, w, lam)
        else:
            args = (X, y, rev_pos, w, lam)

        r = minimize(
            obj, s,
            args=args,
            jac=grad,
            method="L-BFGS-B",
            bounds=[(-BOUND, BOUND)] * len(s),
            options={"maxiter": MAXITER}
        )
        if (best is None) or (r.fun < best.fun):
            best = r
    return best

def align_to_anchor(alpha, beta, se_alpha, se_beta, anchor):
    d1 = np.linalg.norm(beta - anchor)
    d2 = np.linalg.norm(alpha - anchor)
    if d1 <= d2:
        return alpha, beta, se_alpha, se_beta, d1, d2, "as_is"
    else:
        return beta, alpha, se_beta, se_alpha, d2, d1, "swapped"

def extract_shared(best, obj_fun, X, y, w, rev_pos=None, lam=0.3, anchor=None):
    p = X.shape[1]
    zhat = best.x
    _, _, alpha, beta = unpack_md(zhat, p)

    if rev_pos is None:
        H = approx_hess(zhat, lambda z: obj_fun(z, X, y, w, lam))
        raw_nll = raw_shared_obs_nll_from_ab(alpha, beta, X, y, w)
    else:
        H = approx_hess(zhat, lambda z: obj_fun(z, X, y, rev_pos, w, lam))
        raw_nll = raw_shared_pr_nll_from_ab(alpha, beta, X, y, rev_pos, w)

    cov_z = np.linalg.pinv(H)
    J = np.block([[np.eye(p), np.eye(p)],
                  [np.eye(p), -np.eye(p)]])
    cov_ab = J @ cov_z @ J.T
    se_ab = np.sqrt(np.clip(np.diag(cov_ab), 0, np.inf))
    se_alpha = se_ab[:p]
    se_beta = se_ab[p:]

    if anchor is not None:
        alpha, beta, se_alpha, se_beta, d_keep, d_other, how = align_to_anchor(
            alpha, beta, se_alpha, se_beta, anchor
        )
    else:
        d_keep, d_other, how = np.nan, np.nan, "none"

    return {
        "success": bool(best.success),
        "pen_obj": float(best.fun),
        "raw_nll": float(raw_nll),
        "cond_hess": float(np.linalg.cond(H)),
        "alpha": alpha,
        "beta": beta,
        "se_alpha": se_alpha,
        "se_beta": se_beta,
        "anchor_distance_kept": float(d_keep),
        "anchor_distance_other": float(d_other),
        "alignment": how
    }

# -----------------------------
# summary helpers
# -----------------------------
def summarize_nonshared(res, xa_cols, xb_cols, tag):
    ta = pd.DataFrame({
        "block": "alpha",
        "term": xa_cols,
        f"estimate_{tag}": res["alpha"],
        f"se_{tag}": res["se_alpha"],
        f"OR_{tag}": np.exp(res["alpha"])
    })
    tb = pd.DataFrame({
        "block": "beta",
        "term": xb_cols,
        f"estimate_{tag}": res["beta"],
        f"se_{tag}": res["se_beta"],
        f"OR_{tag}": np.exp(res["beta"])
    })
    return pd.concat([ta, tb], ignore_index=True)

def summarize_shared(res, x_cols, tag):
    ta = pd.DataFrame({
        "block": "alpha",
        "term": x_cols,
        f"estimate_{tag}": res["alpha"],
        f"se_{tag}": res["se_alpha"],
        f"OR_{tag}": np.exp(res["alpha"])
    })
    tb = pd.DataFrame({
        "block": "beta",
        "term": x_cols,
        f"estimate_{tag}": res["beta"],
        f"se_{tag}": res["se_beta"],
        f"OR_{tag}": np.exp(res["beta"])
    })
    return pd.concat([ta, tb], ignore_index=True)

# -----------------------------
# load NHANES
# -----------------------------
BASE = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/"
FILES = {
    "DEMO": "P_DEMO.XPT",
    "DIQ":  "P_DIQ.XPT",
    "HIQ":  "P_HIQ.XPT",
    "HUQ":  "P_HUQ.XPT",
    "BMX":  "P_BMX.XPT",
    "GHB":  "P_GHB.XPT",
    "GLU":  "P_GLU.XPT",
}

dfs = {k: read_xpt(BASE + v) for k, v in FILES.items()}
df = dfs["DEMO"].copy()
for k in ["DIQ", "HIQ", "HUQ", "BMX", "GHB", "GLU"]:
    df = df.merge(dfs[k], on="SEQN", how="left")

# -----------------------------
# construct common variables
# -----------------------------
dat = pd.DataFrame()
dat["SEQN"] = df["SEQN"]
dat["AGE_YEARS"] = df["RIDAGEYR"]
dat["FEMALE"] = np.where(df["RIAGENDR"] == 2, 1.0,
                  np.where(df["RIAGENDR"] == 1, 0.0, np.nan))
dat["BMI"] = df["BMXBMI"]

# self-reported diabetes
dat["Y"] = np.where(df["DIQ010"] == 1, 1.0,
             np.where(df["DIQ010"] == 2, 0.0, np.nan))

# insurance
dat["INSURED"] = np.where(df["HIQ011"] == 1, 1.0,
                   np.where(df["HIQ011"] == 2, 0.0, np.nan))

# usual source of care
huq = df["HUQ030"]
dat["USUALCARE"] = np.select(
    [huq.isin([1, 2]), huq == 3],
    [1.0, 0.0],
    default=np.nan
)

# biomarkers
dat["D_A1C"] = np.where(df["LBXGH"].notna(), (df["LBXGH"] >= 6.5).astype(float), np.nan)
dat["D_FPG"] = np.where(df["LBXGLU"].notna(), (df["LBXGLU"] >= 126.0).astype(float), np.nan)

# weights
dat["WTMECPRP"] = df["WTMECPRP"]
dat["WTSAFPRP"] = df["WTSAFPRP"]

# adults
dat = dat[dat["AGE_YEARS"] >= ADULT_AGE].copy()

# -----------------------------
# run A1C primary and FPG sensitivity
# -----------------------------
for MODE in ["A1C", "FPG"]:
    if MODE == "A1C":
        DVAR = "D_A1C"
        WVAR = "WTMECPRP"
        base = dat.dropna(subset=["Y", "INSURED", "USUALCARE", "AGE_YEARS", "BMI", "FEMALE", WVAR]).copy()
        base = base[base[WVAR] > 0].copy()
    else:
        DVAR = "D_FPG"
        WVAR = "WTSAFPRP"
        base = dat.dropna(subset=["Y", "INSURED", "USUALCARE", "AGE_YEARS", "BMI", "FEMALE", WVAR]).copy()
        base = base[base[WVAR] > 0].copy()

    # standardize within analysis base
    base["AGE_YEARS_Z"] = (base["AGE_YEARS"] - base["AGE_YEARS"].mean()) / base["AGE_YEARS"].std(ddof=0)
    base["BMI_Z"] = (base["BMI"] - base["BMI"].mean()) / base["BMI"].std(ddof=0)
    base["const"] = 1.0

    # one-sided partial reveals
    base["R"] = np.where((base["Y"] == 0) & base[DVAR].notna(), 1.0, 0.0)
    base["REV_POS"] = np.where((base["Y"] == 0) & (base[DVAR] == 1), 1.0, 0.0)

    # arrays
    Xa = base[XA_COLS].to_numpy(dtype=float)
    Xb = base[XB_COLS].to_numpy(dtype=float)
    Xs = base[XS_COLS].to_numpy(dtype=float)
    y = base["Y"].to_numpy(dtype=float)
    rev_pos = base["REV_POS"].to_numpy(dtype=float)
    w_raw = base[WVAR].to_numpy(dtype=float)
    w = w_raw / np.mean(w_raw)

    # descriptive summaries
    desc = {
        "mode": MODE,
        "weight_variable": WVAR,
        "n_base": int(len(base)),
        "n_y1": int(np.sum(y == 1)),
        "n_y0": int(np.sum(y == 0)),
        "n_y0_r1": int(np.sum(base["R"].values == 1)),
        "n_y0_revpos1": int(np.sum(rev_pos == 1)),
        "weighted_prev_self_report": float(weighted_mean(base["Y"].values, w_raw)),
        "weighted_prev_proxy_among_observed": float(
            weighted_mean(base.loc[base[DVAR].notna(), DVAR].values,
                          base.loc[base[DVAR].notna(), WVAR].values)
        ) if np.sum(base[DVAR].notna()) > 0 else np.nan,
        "weighted_prop_y0_given_proxy1": float(
            weighted_mean(
                base.loc[(base[DVAR] == 1), "Y"].apply(lambda t: 1.0 if t == 0 else 0.0).values,
                base.loc[(base[DVAR] == 1), WVAR].values
            )
        ) if np.sum(base[DVAR] == 1) > 0 else np.nan,
        "weighted_prop_proxy1_among_y0_observed": float(
            weighted_mean(
                base.loc[(base["Y"] == 0) & base[DVAR].notna(), DVAR].values,
                base.loc[(base["Y"] == 0) & base[DVAR].notna(), WVAR].values
            )
        ) if np.sum((base["Y"] == 0) & base[DVAR].notna()) > 0 else np.nan
    }

    print(f"\n==================== {MODE} : DESCRIPTIVE ====================")
    print(pd.Series(desc).to_string())

    # ---------- non-shared ----------
    fit_obs_ns = fit_obs_nonshared(Xa, Xb, y, w)
    fit_pr_ns  = fit_pr_nonshared(Xa, Xb, y, rev_pos, w)

    tab_obs_ns = summarize_nonshared(fit_obs_ns, XA_COLS, XB_COLS, "ZI_NS")
    tab_pr_ns  = summarize_nonshared(fit_pr_ns,  XA_COLS, XB_COLS, "PR_NS")
    tab_ns = tab_obs_ns.merge(tab_pr_ns, on=["block", "term"], how="outer")
    tab_ns["SE_ratio_PR_over_ZI"] = tab_ns["se_PR_NS"] / tab_ns["se_ZI_NS"]

    diag_ns = pd.DataFrame([
        {
            "model": "OBS_ZI_nonshared",
            "success": fit_obs_ns["success"],
            "nll": fit_obs_ns["nll"],
            "cond_hess": fit_obs_ns["cond_hess"]
        },
        {
            "model": "PR_nonshared",
            "success": fit_pr_ns["success"],
            "nll": fit_pr_ns["nll"],
            "cond_hess": fit_pr_ns["cond_hess"]
        }
    ])

    print(f"\n==================== {MODE} : NON-SHARED DIAGNOSTICS ====================")
    print(diag_ns.to_string(index=False))
    print(f"\n==================== {MODE} : NON-SHARED COEFFICIENT TABLE ====================")
    print(tab_ns.round(4).to_string(index=False))

    # ---------- shared ----------
    anchor = fit_weighted_logit(Xs, y, w)
    starts_shared = make_starts_shared(anchor, n_starts=N_STARTS_SHARED, jitter_sd=JITTER_SD_SHARED, seed=SEED)

    best_obs_sh = fit_multistart_shared(
        shared_obs_nll, shared_obs_grad, Xs, y, w,
        starts=starts_shared, rev_pos=None, lam=RIDGE_LAMBDA_SHARED
    )
    res_obs_sh = extract_shared(
        best_obs_sh, shared_obs_nll, Xs, y, w,
        rev_pos=None, lam=RIDGE_LAMBDA_SHARED, anchor=anchor
    )

    starts_pr_shared = [best_obs_sh.x] + starts_shared
    best_pr_sh = fit_multistart_shared(
        shared_pr_nll, shared_pr_grad, Xs, y, w,
        starts=starts_pr_shared, rev_pos=rev_pos, lam=RIDGE_LAMBDA_SHARED
    )
    res_pr_sh = extract_shared(
        best_pr_sh, shared_pr_nll, Xs, y, w,
        rev_pos=rev_pos, lam=RIDGE_LAMBDA_SHARED, anchor=anchor
    )

    tab_obs_sh = summarize_shared(res_obs_sh, XS_COLS, "ZI_SH")
    tab_pr_sh  = summarize_shared(res_pr_sh,  XS_COLS, "PR_SH")
    tab_sh = tab_obs_sh.merge(tab_pr_sh, on=["block", "term"], how="outer")
    tab_sh["SE_ratio_PR_over_ZI"] = tab_sh["se_PR_SH"] / tab_sh["se_ZI_SH"]

    diag_sh = pd.DataFrame([
        {
            "model": "OBS_ZI_shared",
            "success": res_obs_sh["success"],
            "pen_obj": res_obs_sh["pen_obj"],
            "raw_nll": res_obs_sh["raw_nll"],
            "cond_hess": res_obs_sh["cond_hess"],
            "anchor_distance_kept": res_obs_sh["anchor_distance_kept"],
            "anchor_distance_other": res_obs_sh["anchor_distance_other"],
            "alignment": res_obs_sh["alignment"]
        },
        {
            "model": "PR_shared",
            "success": res_pr_sh["success"],
            "pen_obj": res_pr_sh["pen_obj"],
            "raw_nll": res_pr_sh["raw_nll"],
            "cond_hess": res_pr_sh["cond_hess"],
            "anchor_distance_kept": res_pr_sh["anchor_distance_kept"],
            "anchor_distance_other": res_pr_sh["anchor_distance_other"],
            "alignment": res_pr_sh["alignment"]
        }
    ])

    print(f"\n==================== {MODE} : SHARED DIAGNOSTICS ====================")
    print(diag_sh.to_string(index=False))
    print(f"\n==================== {MODE} : SHARED COEFFICIENT TABLE ====================")
    print(tab_sh.round(4).to_string(index=False))

    # summaries
    alpha_mean_ns = float(np.nanmean(tab_ns.loc[tab_ns["block"] == "alpha", "SE_ratio_PR_over_ZI"]))
    beta_mean_ns  = float(np.nanmean(tab_ns.loc[tab_ns["block"] == "beta",  "SE_ratio_PR_over_ZI"]))
    alpha_mean_sh = float(np.nanmean(tab_sh.loc[tab_sh["block"] == "alpha", "SE_ratio_PR_over_ZI"]))
    beta_mean_sh  = float(np.nanmean(tab_sh.loc[tab_sh["block"] == "beta",  "SE_ratio_PR_over_ZI"]))

    block_summary = pd.DataFrame([
        {"comparison": "nonshared", "alpha_block_mean_ratio": alpha_mean_ns, "beta_block_mean_ratio": beta_mean_ns},
        {"comparison": "shared",    "alpha_block_mean_ratio": alpha_mean_sh, "beta_block_mean_ratio": beta_mean_sh},
    ])

    print(f"\n==================== {MODE} : BLOCKWISE SE-RATIO SUMMARY ====================")
    print(block_summary.to_string(index=False))

    # save
    pd.Series(desc).to_csv(os.path.join(OUTDIR, f"desc_{MODE.lower()}.csv"))
    tab_ns.to_csv(os.path.join(OUTDIR, f"coef_nonshared_{MODE.lower()}.csv"), index=False)
    tab_sh.to_csv(os.path.join(OUTDIR, f"coef_shared_{MODE.lower()}.csv"), index=False)
    diag_ns.to_csv(os.path.join(OUTDIR, f"diag_nonshared_{MODE.lower()}.csv"), index=False)
    diag_sh.to_csv(os.path.join(OUTDIR, f"diag_shared_{MODE.lower()}.csv"), index=False)
    block_summary.to_csv(os.path.join(OUTDIR, f"block_summary_{MODE.lower()}.csv"), index=False)

print(f"\nSaved outputs to: {OUTDIR}")


==================== A1C : DESCRIPTIVE ====================
mode                                           A1C
weight_variable                           WTMECPRP
n_base                                        8114
n_y1                                          1283
n_y0                                          6831
n_y0_r1                                       6463
n_y0_revpos1                                   220
weighted_prev_self_report                 0.117831
weighted_prev_proxy_among_observed        0.098987
weighted_prop_y0_given_proxy1             0.190231
weighted_prop_proxy1_among_y0_observed    0.021378

==================== A1C : NON-SHARED DIAGNOSTICS ====================
           model  success         nll    cond_hess
OBS_ZI_nonshared     True 2432.597926 12502.743624
    PR_nonshared     True 2454.084235 30101.950835

==================== A1C : NON-SHARED COEFFICIENT TABLE ====================
block        term  estimate_ZI_NS  se_ZI_NS  OR_ZI_NS  estimate_PR_NS  se_P